# Cambodian ALPR — run #4: make the STN actually straighten

## The problem this run fixes

The "straightening layer" (STN) was supposed to rotate an upside-down plate
upright before the reader sees it. Measured on 2026-08-20, **it never learned to
do that.** On `crnn_stn2`, `crnn_stn4` and `crnn_stn5` alike it predicts

```
theta = [[+1.09,  0.00, +0.05]        a 180 flip would be   [[-1, 0, 0]
         [ 0.00, +1.05, -0.01]]                              [ 0,-1, 0]]
```

— a near-identity transform, essentially the same for flipped and upright input.
It cannot even tell them apart. So the CRNN was reading flipped text directly,
and one set of weights had to cover two different reading tasks. That is why
upside-down sits at 51.7% while upright is at 87.2%.

## The fix

During training **we** are the code that flips the crop (`--rotate180 0.5`), so
we already know the right answer. The new `--stn-supervise W` flag adds a loss
term that teaches the STN directly:

* crop was flipped  -> predict exactly `[[-1,0,0],[0,-1,0]]`
* crop was upright  -> predict exactly `[[+1,0,0],[0,+1,0]]`

Targeting the exact matrix also removes the stray ~9% zoom it drifted into.

If it works, the STN straightens the plate and the CRNN reads it with the same
ability that already gives 87% upright.

**Target: upside-down 51.7% -> 70s. Ceiling is 87.2%** (54 of the 149 test crops
read perfectly upright and fail only when flipped; only 18 are truly unreadable).

Honest note: this is a projection, not a promise. The measure cells decide.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BUNDLE = '/content/drive/MyDrive/ALPR/alpr_colab_bundle.zip'

import os, zipfile, shutil
shutil.rmtree('/content/alpr', ignore_errors=True)
os.makedirs('/content/alpr', exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall('/content/alpr')
%cd /content/alpr

In [ ]:
# ---- BUNDLE FRESHNESS CHECK -- do not skip -------------------------------
import csv, collections, os

ft   = open('scripts/recognition/finetune_crnn.py', encoding='utf-8').read()
crnn = open('src/recognition/crnn_model.py', encoding='utf-8').read()

ok_sup   = 'stn-supervise' in ft and 'stn_target_theta' in ft
ok_theta = 'def theta' in crnn
ok_tool  = os.path.exists('scripts/tools/check_stn.py')
ok_clean = os.path.exists('data/crnn_crops/real_labels_deleaked.csv')

print('--stn-supervise flag :', 'PRESENT' if ok_sup   else '*** MISSING ***')
print('STN.theta() exposed  :', 'PRESENT' if ok_theta else '*** MISSING ***')
print('check_stn.py         :', 'PRESENT' if ok_tool  else '*** MISSING ***')
print('de-leaked labels     :', 'PRESENT' if ok_clean else '*** MISSING ***')
assert ok_sup and ok_theta and ok_tool and ok_clean, 'STALE BUNDLE -- rebuild and re-upload.'

c = collections.Counter(r['image_path'].replace(chr(92), '/').split('/')[2]
                        for r in csv.DictReader(
                            open('data/crnn_crops/real_labels_deleaked.csv', encoding='utf-8')))
print()
print('de-leaked labels:', dict(c))
assert c['train'] == 716 and c['valid'] == 157 and c['test'] == 149, 'wrong label file'
print()
print('Bundle is current. Safe to continue.')

In [ ]:
!pip -q install ultralytics opencv-python-headless pyyaml tqdm pillow matplotlib

In [ ]:
!python scripts/recognition/generate_synthetic.py --train 20000 --valid 1500 --test 800

In [ ]:
# THE EXPERIMENT -- same as crnn_stn5, plus the supervised straightening loss.
!python scripts/recognition/finetune_crnn.py --stn --stn-supervise 1.0 --rotate180 0.5 \
    --synth-n 16000 --epochs 80 \
    --out models/recognition/crnn_stn6.pth \
    --real-csv data/crnn_crops/real_labels_deleaked.csv \
    --extra-real-csv data/crnn_crops/real_labels_deleaked.csv \
    --extra-match /valid/ --extra-real-oversample 6

Watch the new **`stn`** number in the epoch lines. It starts around 0.60 (the
layer is at identity, so it is wrong on every flipped crop) and should fall
toward 0. If it is still near 0.6 at epoch 20, the layer is not learning the
orientation and the next two cells will say FAIL.

In [ ]:
# DID THE STN LEARN TO ROTATE? -- this is the mechanism check.
!python scripts/tools/check_stn.py --weights models/recognition/crnn_stn6.pth

In [ ]:
# MEASURE -- headline, all 149 held-out crops. Compare against crnn_stn5: 87.2% / 51.7%
!python scripts/tools/test_rotation_reading.py --weights models/recognition/crnn_stn6.pth

In [ ]:
# Bucket breakdown, for continuity with the earlier runs.
!python scripts/tools/check_leakage.py --audit --weights models/recognition/crnn_stn6.pth

### How to read it

Two separate questions — answer both:

**1. Did the mechanism work?** (`check_stn.py`)

| result | meaning |
|---|---|
| PASS | the STN genuinely straightens. The write-up can claim it, with proof |
| FAIL | still decorative — the supervision loss did not take |

**2. Did accuracy improve?** (`test_rotation_reading.py`, vs 87.2% up / 51.7% down)

| result | meaning |
|---|---|
| upside-down well above 51.7%, upright ~87% | the fix worked — this is the new best model |
| upside-down up, upright down | the STN is over-rotating upright crops; lower `--stn-supervise` to 0.3 |
| both roughly unchanged | straightening was not the bottleneck; the CRNN was already coping |
| both worse | the aux loss is fighting the reading loss; lower the weight |

Note PASS on question 1 with no gain on question 2 is a perfectly possible — and
still reportable — outcome: it would mean the reader never needed the help.

Paste the output of the last three cells back to Claude in full.

In [ ]:
import shutil, os
os.makedirs('/content/drive/MyDrive/ALPR/trained', exist_ok=True)
p = 'models/recognition/crnn_stn6.pth'
if os.path.exists(p):
    shutil.copy(p, '/content/drive/MyDrive/ALPR/trained/')
    print('saved crnn_stn6.pth -> Drive/ALPR/trained')
else:
    print('crnn_stn6.pth was not produced -- training failed, check above.')